# Ablation: một lần chạy = một cấu hình = một ZIP riêng
Chọn ABLATION_CONFIG ở cell cấu hình rồi Run All. Mặc định chạy đủ 120 câu của DUY NHẤT cấu hình đó, dùng Gemini 3.1 cho vai trò Gemini trong pipeline và Gemini 3.5 Flash-Lite để chấm sau. Qwen3:8b/Ollama giữ vai trò như cấu hình thí nghiệm. Không tự chạy bốn cấu hình còn lại.

Ví dụ: full → ZIP full; without_graph → ZIP without_graph. Đo từng cấu hình xong mới gom ZIP bằng compare_ablation_results.py để lập bảng so sánh.

ACTION=run_and_judge: chạy rồi chấm. generate_only: chỉ tạo đáp án. judge_only: chỉ chấm các đáp án đã có, không chạy lại pipeline. Có thể đặt MAX_NEW_QUESTIONS/MAX_NEW_JUDGMENTS thành số dương để tạm chia lô; mặc định None nghĩa là chạy hết cấu hình đã chọn.

120 câu không đồng nghĩa 120 request Gemini; một câu có thể gọi nhiều lần. Hết quota thì dừng và tải checkpoint, không tự xoay key/model. Khi có quota lại, giữ nguyên cấu hình/model/code và chạy tiếp; chỉ bật RETRY_ERRORS nếu cần đo lại ERROR. Tải ZIP trước khi phiên Kaggle kết thúc.

GPU_SPEC cho phép khác UUID GPU nhưng phải giữ cùng loại/VRAM/driver và các điều kiện CPU/package/code/model được kiểm soát. UUID thật vẫn được ghi; không khẳng định cùng thiết bị vật lý hoặc độ trễ cloud giống hệt. Pin CODE_REF bằng SHA đã dùng để resume. Không thay fingerprint để ép gộp kết quả khác điều kiện.

Lần đo timeoutguard v2 dùng RUN_NAME mới. Không resume/trộn với kết quả cũ sau khi sửa code/config.
Preflight tải Qwen lên GPU với context 32K bằng một câu tổng hợp cố định, không dùng Gold. Thời gian này được ghi riêng.
Auditor chỉ gọi khi đủ cửa sổ thời gian tối thiểu, chia đợt theo kích thước prompt, không cắt nội dung luật.
Nếu đã lưu đủ lượt nhưng còn ERROR, notebook vẫn chấm các lượt OK và giữ nguyên mẫu số/lỗi trong ZIP.
Giữ cùng BACKEND_GEMINI_MODEL và JUDGE_GEMINI_MODEL cho cả năm cấu hình. Đổi judge cần đợt chấm riêng cho các baseline để so sánh.


In [ ]:
from pathlib import Path
import re
REPO_URL='https://github.com/tvqui/legal-agentic-rag.git'
CODE_REF='origin/main'  # use the printed commit SHA for a later session
ROOT=Path('/kaggle/working/legal-agentic-rag')
WORK=Path('/kaggle/working/ablation_evaluation')
RUN_NAME='labor120_ablation_timeoutguard_v2'
PROFILE='standard'  # smoke = 5 fixed questions; standard = all 120
ARTIFACT='auto'
LABOR_GOLD='gold_dataset/PhapDienAI_Gold_120_Labor_v0_2.zip'
BACKEND_GEMINI_MODEL='gemini-3.1-flash-lite'
JUDGE_GEMINI_MODEL='gemini-3.5-flash-lite'
BACKEND_GEMINI_SECRET_NAME='GEMINI_API_KEY'
JUDGE_GEMINI_SECRET_NAME='GEMINI_API_KEY'  # optionally GEMINI_API_KEY.2, .3; no automatic rotation
ABLATION_CONFIG='full'  # full | without_graph | without_reranker | without_auditor | without_review
ACTION='run_and_judge'  # run_and_judge | generate_only | judge_only
MAX_NEW_QUESTIONS=None     # new answer attempts, NOT number of API requests
MAX_NEW_JUDGMENTS=None
CLEANUP_OLLAMA=True    # keep the owned server ready between independent cells
RETRY_ERRORS=False      # intentionally enable after capacity is restored
CHECKPOINT_ARCHIVE=''   # optional downloaded ablation ZIP added as Input
METHODS=('full','without_graph','without_reranker','without_auditor','without_review')
PHASE='ablation';STAGES=['ablation']
if PROFILE not in {'smoke','standard'}:raise ValueError('Invalid PROFILE')
if not re.fullmatch(r'[A-Za-z0-9_-]+',RUN_NAME):raise ValueError('Invalid RUN_NAME')
for value in [BACKEND_GEMINI_SECRET_NAME,JUDGE_GEMINI_SECRET_NAME]:
    if not isinstance(value,str) or not re.fullmatch(r'GEMINI_API_KEY(?:\.[1-9][0-9]*)?',value):raise ValueError('Use Secret NAME, not key value')
if ABLATION_CONFIG not in METHODS:raise ValueError('Invalid ABLATION_CONFIG')
if ACTION not in {'run_and_judge','generate_only','judge_only'}:raise ValueError('Invalid ACTION')
for limit in [MAX_NEW_QUESTIONS,MAX_NEW_JUDGMENTS]:
    if limit is not None and (type(limit) is not int or limit<1):raise ValueError('Batch limit must be positive or None')
OUT=WORK/RUN_NAME/PROFILE/ABLATION_CONFIG

## 1. Lấy code GitHub và cài môi trường

In [ ]:
import os,sys,subprocess,shutil,json,time,urllib.request
WORK.mkdir(parents=True,exist_ok=True)
def run(args, **kw):
    print('RUN',args[0],flush=True)
    return subprocess.run(args,check=True,**kw)
run(['apt-get','update','-y'])
run(['apt-get','install','-y','git','curl','zstd'])
if not ROOT.exists():
    run(['git','clone','--branch','main','--single-branch',REPO_URL,str(ROOT)])
else:
    dirty=subprocess.check_output(['git','status','--porcelain','--untracked-files=no'],cwd=ROOT,text=True)
    if dirty.strip():raise RuntimeError('Tracked code was edited in the notebook. Save changes before updating.')
    run(['git','fetch','--prune','origin'],cwd=ROOT)
    run(['git','checkout','--detach','origin/main'],cwd=ROOT)
run(['git','fetch','--prune','origin'],cwd=ROOT)
run(['git','checkout','--detach',CODE_REF],cwd=ROOT)
run(['git','log','-1','--oneline'],cwd=ROOT)
# Keep Kaggle's CUDA-compatible Torch and isolate project dependencies.
import importlib.metadata
constraints=[]
for package in ('torch','torchvision','torchaudio'):
    try:constraints.append(package+'=='+importlib.metadata.version(package))
    except importlib.metadata.PackageNotFoundError:
        if package=='torch':raise RuntimeError('Kaggle runtime has no Torch; select a GPU image.')
constraint_file=WORK/'torch_constraints.txt'
constraint_file.write_text('\n'.join(constraints)+'\n',encoding='utf-8')
VENV=WORK/'.venv'
run([sys.executable,'-m','venv','--system-site-packages','--without-pip',str(VENV)])
PYTHON=str(VENV/'bin/python')
run([sys.executable,'-m','pip','--python',PYTHON,'install','--disable-pip-version-check',
    '-q','-c',str(constraint_file),'-e',str(ROOT)+'[retrieval,online]','datasets'])
run([PYTHON,'-c','import torch,FlagEmbedding,faiss,datasets; assert torch.cuda.is_available(); print("CUDA dependencies ready:",torch.cuda.device_count())'])
if not (ROOT/'benchmarks/evaluation_suite/protocol.py').exists() or 'labor-evaluation-v2-20261028' not in (ROOT/'benchmarks/evaluation_suite/protocol.py').read_text(encoding='utf-8'):
    raise RuntimeError('Push evaluation suite v2 before running this notebook.')
if not (ROOT/'benchmarks/evaluation_suite/suite.py').exists():
    raise RuntimeError('Push the evaluation_suite folder to GitHub before Run All.')

if not (ROOT/'đo/evaluate_project.py').exists():
    raise RuntimeError('Chưa có bộ đo mới trên GitHub. Push folder đo rồi chạy lại cell lấy code.')
run([sys.executable,'-m','pip','--python',PYTHON,'install','--disable-pip-version-check','-q',
     '-c',str(constraint_file),'-r',str(ROOT/'đo/requirements.txt')])
engine=(ROOT/'đo/eval_core.py').read_text(encoding='utf8')
if '--max-new-questions' not in engine or '--max-new-judgments' not in engine:
    raise RuntimeError('Push the new ablation evaluation code before running this notebook')
# Stop early if GitHub still supplies the old timeout policy.
if ACTION!='judge_only':
    guard_probe="from pathlib import Path;import inspect;from vn_labor_online.config import load_config;from vn_labor_online.runtime import operation_budget;c=load_config(Path('config/online_kaggle.yaml'));assert 'minimum_call_seconds' in inspect.signature(operation_budget).parameters;assert c.applicability.context_tokens==c.adjudication.context_tokens==32768;assert c.applicability.minimum_call_seconds==25 and c.applicability.batch_prompt_tokens==7000;assert c.applicability.max_output_tokens==1536;assert c.adjudication.minimum_call_seconds==20;print('TIMEOUT_POLICY_VERIFIED')"
    try:run([PYTHON,'-c',guard_probe],cwd=ROOT)
    except subprocess.CalledProcessError:
        raise RuntimeError('Timeout policy preflight failed; read the Python error above. Generation has not started. Verify code/config before retrying.') from None


In [ ]:
from kaggle_secrets import UserSecretsClient
try:
    hf_token=UserSecretsClient().get_secret('HF_TOKEN')
    if hf_token:os.environ['HF_TOKEN']=hf_token
except Exception:print('HF_TOKEN optional; public downloads will be attempted.')
print('Gemini credentials are loaded only for the selected generation/judge cell; values are never printed.')

## 2. Khởi động Ollama trên GPU 1

In [ ]:
if ACTION!='judge_only':
    if {'ablation','vlsp'} & set(STAGES):
        # Use GPU 1 for Ollama, GPU 0 for embedding/reranker. Own only the server
        # started by this notebook; never terminate another session's server.
        gpu_lines=subprocess.check_output(['nvidia-smi','--query-gpu=name','--format=csv,noheader'],text=True).splitlines()
        print('GPUs:',gpu_lines,flush=True)
        if len(gpu_lines)<2:raise RuntimeError('Select GPU T4 x2 in Kaggle settings for this configuration.')
        if shutil.which('ollama') is None:
            run(['bash','-lc','set -o pipefail; curl -fsSL https://ollama.com/install.sh | sh'])
        def ollama_json(path):
            with urllib.request.urlopen('http://127.0.0.1:11434'+path,timeout=5) as response:return json.load(response)
        OWN_OLLAMA=globals().get('OWN_OLLAMA')
        previous_log=globals().get('OLLAMA_LOG')
        if previous_log is not None and not previous_log.closed:previous_log.close()
        OLLAMA_LOG=(WORK/'ollama.log').open('a',encoding='utf-8')
        try:
            ollama_json('/api/tags')
            print('Reusing existing Ollama; verify it is not serving another workload.',flush=True)
        except Exception:
            environment=dict(os.environ);environment['CUDA_VISIBLE_DEVICES']='1'
            OWN_OLLAMA=subprocess.Popen(['ollama','serve'],env=environment,stdout=OLLAMA_LOG,stderr=subprocess.STDOUT)
            for attempt in range(60):
                if OWN_OLLAMA.poll() is not None:raise RuntimeError('Ollama stopped; read '+str(WORK/'ollama.log'))
                try:ollama_json('/api/tags');break
                except Exception:time.sleep(2)
            else:raise RuntimeError('Ollama did not become ready')
        models={m.get('name') for m in ollama_json('/api/tags').get('models',[])}
        if 'qwen3:8b' not in models:
            for attempt in range(3):
                try:run(['ollama','pull','qwen3:8b'],timeout=900);break
                except (subprocess.CalledProcessError,subprocess.TimeoutExpired):
                    if attempt==2:raise
                    time.sleep(10)
        def warm_ollama():
            # Fixed synthetic input; no Gold, no legal content. Startup latency
            # is recorded separately and excluded from per-question latency.
            payload={'model':'qwen3:8b','stream':False,'think':False,'keep_alive':'15m',
                'format':{'type':'object','properties':{'ready':{'type':'boolean'}},'required':['ready']},
                'options':{'temperature':0,'num_predict':32,'num_ctx':32768},
                'messages':[{'role':'user','content':'Return a JSON object with ready equal to true.'}]}
            request=urllib.request.Request('http://127.0.0.1:11434/api/chat',
                data=json.dumps(payload).encode('utf-8'),headers={'Content-Type':'application/json'})
            started=time.monotonic()
            try:
                with urllib.request.urlopen(request,timeout=180) as response:
                    result=json.load(response)
            except Exception:
                raise RuntimeError('Ollama preflight failed. Do not retry generation while an old inference may still be running; inspect ollama.log and restart the owned server/session.') from None
            content=json.loads(result.get('message',{}).get('content','{}'))
            if not result.get('done') or result.get('done_reason')=='length' or content!={'ready':True}:
                raise RuntimeError('Ollama preflight output is incomplete or invalid.')
            loaded=[m for m in ollama_json('/api/ps').get('models',[]) if m.get('name',m.get('model'))=='qwen3:8b']
            if not loaded or not loaded[0].get('size_vram',0)>0:
                raise RuntimeError('Qwen is not resident on GPU; inspect GPU/VRAM before starting 120 questions.')
            status={'status':'READY','model':'qwen3:8b','context_tokens':32768,
                'size_vram':loaded[0]['size_vram'],'size':loaded[0].get('size'),
                'elapsed_seconds':time.monotonic()-started,'scope':'SYNTHETIC_STARTUP_NOT_QUERY_LATENCY'}
            (WORK/'ollama_preflight.json').write_text(json.dumps(status,indent=2),encoding='utf-8')
            print('Ollama GPU preflight:',status,flush=True)
            return status
        warm_ollama()
        print('Ollama and model ready.',flush=True)
    
    else:
        OWN_OLLAMA=None;OLLAMA_LOG=None
else:
    OWN_OLLAMA=None;OLLAMA_LOG=None
    print('Judge-only: no Ollama/model-generation startup required.')


## 3. Chuẩn bị hoặc khôi phục checkpoint

In [ ]:
import json,sys,shutil,zipfile,stat
from pathlib import PurePosixPath
if CHECKPOINT_ARCHIVE and not OUT.exists():
    with zipfile.ZipFile(CHECKPOINT_ARCHIVE) as archive:
        if sum(m.file_size for m in archive.infolist())>8*1024**3:raise RuntimeError('Checkpoint too large')
        for m in archive.infolist():
            name=PurePosixPath(m.filename)
            if name.is_absolute() or '..' in name.parts or chr(92) in m.filename or ':' in m.filename or stat.S_ISLNK(m.external_attr>>16):raise RuntimeError('Unsafe checkpoint path')
        OUT.mkdir(parents=True)
        archive.extractall(OUT)
OUT.mkdir(parents=True,exist_ok=True)
BUNDLE=OUT/'bundle120';SETTINGS=OUT/'settings.json'
if ACTION!='judge_only' and (WORK/'ollama_preflight.json').exists():
    shutil.copyfile(WORK/'ollama_preflight.json',OUT/'ollama_preflight.json')
sys.path.insert(0,str(ROOT/'benchmarks/evaluation_suite'))
from common import detect_artifact
artifact=Path(detect_artifact(Path('/kaggle/input'))) if ARTIFACT=='auto' else Path(ARTIFACT)
config=json.loads((ROOT/'đo/settings.example.json').read_text(encoding='utf8'))
config.update(online_config=str(ROOT/'config/online_kaggle.yaml'),cache=str(WORK/'cache'),device='cuda:0')
config['judge']['model']=JUDGE_GEMINI_MODEL
config['hardware_comparison_scope']='GPU_SPEC'  # same GPU model/VRAM/driver; actual UUIDs remain logged
if SETTINGS.exists() and json.loads(SETTINGS.read_text(encoding='utf8'))!=config:raise RuntimeError('Settings changed; use NEW RUN_NAME')
SETTINGS.write_text(json.dumps(config,ensure_ascii=False,indent=2),encoding='utf8')
selection={'backend_gemini_model':BACKEND_GEMINI_MODEL,'judge_gemini_model':JUDGE_GEMINI_MODEL,'ablation_config':ABLATION_CONFIG}
pinned=OUT/'experiment_config.json'
if pinned.exists() and json.loads(pinned.read_text(encoding='utf8'))!=selection:raise RuntimeError('Model plan changed; use NEW RUN_NAME')
pinned.write_text(json.dumps(selection,indent=2),encoding='utf8')
if not (BUNDLE/'manifest.json').exists():
    run([PYTHON,'-u','đo/evaluate_project.py','prepare','--artifact',str(artifact),'--gold',str(ROOT/LABOR_GOLD),'--bundle',str(BUNDLE),'--cache',str(WORK/'cache'),'--allow-draft'],cwd=ROOT)
else:
    frozen=json.loads((BUNDLE/'manifest.json').read_text(encoding='utf8'))
    if Path(frozen['artifact_source']).resolve()!=artifact.resolve():raise RuntimeError('Artifact path changed; do not silently rebind checkpoint')
print('Bundle frozen:',BUNDLE)


## 4. Hàm điều khiển — không tự gọi model

In [ ]:
from contextlib import contextmanager
from IPython.display import FileLink,display

@contextmanager
def model_environment(stage):
    from kaggle_secrets import UserSecretsClient
    label=BACKEND_GEMINI_SECRET_NAME if stage=='generation' else JUDGE_GEMINI_SECRET_NAME
    try:token=UserSecretsClient().get_secret(label)
    except Exception:raise RuntimeError('Enable selected Secret: '+label) from None
    if not isinstance(token,str) or not token.strip():raise RuntimeError('Selected Secret is empty')
    values={'GEMINI_API_KEY':token.strip(),'GEMINI_MODEL':BACKEND_GEMINI_MODEL,
        'VN_LABOR_GEMINI_RPM':'13','VN_LABOR_GEMINI_MAX_RETRIES':'2','VN_LABOR_GEMINI_MAX_WAIT_SECONDS':'70',
        'VN_LABOR_RESEARCHER_API_KEY':None,'VN_LABOR_APPLICABILITY_API_KEY':None,'VN_LABOR_ADJUDICATION_API_KEY':None,
        'VN_LABOR_RESEARCHER_URL':'http://127.0.0.1:11434/api/chat',
        'VN_LABOR_APPLICABILITY_MODE':'hybrid','VN_LABOR_APPLICABILITY_PROVIDER':'ollama',
        'VN_LABOR_APPLICABILITY_MODEL':'qwen3:8b','VN_LABOR_APPLICABILITY_URL':'http://127.0.0.1:11434/api/chat',
        'VN_LABOR_ADJUDICATION_URL':'http://127.0.0.1:11434/api/chat',
        'VN_LABOR_RESEARCHER_MODE':'gemini','VN_LABOR_RESEARCHER_MODEL':BACKEND_GEMINI_MODEL,
        'VN_LABOR_ADJUDICATION_MODE':'ollama','VN_LABOR_ADJUDICATION_MODEL':'qwen3:8b',
        'VN_LABOR_ADJUDICATION_REVIEW_PROVIDER':'gemini','VN_LABOR_ADJUDICATION_REVIEW_MODEL':BACKEND_GEMINI_MODEL}
    previous={name:os.environ.get(name) for name in values}
    try:
        for name,value in values.items():
            if value is None:os.environ.pop(name,None)
            else:os.environ[name]=value
        yield
    finally:
        for name,value in previous.items():
            if value is None:os.environ.pop(name,None)
            else:os.environ[name]=value

def export_checkpoint():
    archive=Path(shutil.make_archive(str(OUT)+'_ablation_results','zip',root_dir=OUT))
    display(FileLink(str(archive)))
    return archive

def run_configuration(method):
    if method not in METHODS:raise ValueError('Invalid method')
    if method!=ABLATION_CONFIG or ACTION=='judge_only':
        print('Generation not selected:',method);return
    destination=OUT/'runs'/method
    args=[PYTHON,'-u','đo/evaluate_project.py','run','--bundle',str(BUNDLE),'--config',str(SETTINGS),
          '--out',str(destination),'--task','qa','--methods',method,'--repeats','1','--seed','42','--k','10','--allow-draft','--allow-llm']
    if PROFILE=='smoke':args+=['--limit','5']
    if (destination/'run_manifest.json').exists():args+=['--resume']
    if RETRY_ERRORS:args+=['--retry-errors']
    if MAX_NEW_QUESTIONS is not None:args+=['--max-new-questions',str(MAX_NEW_QUESTIONS)]
    try:
        with model_environment('generation'):result=subprocess.run(args,cwd=ROOT,check=False)
        progress=json.loads((destination/'progress.json').read_text(encoding='utf8')) if (destination/'progress.json').exists() else {}
        print(method,'progress:',progress)
        scheduled=(type(progress.get('expected')) is int and progress['expected']>0
            and progress.get('saved')==progress['expected'] and progress.get('paused') is None)
        known_checkpoint=result.returncode==2 and (progress.get('paused')=='BATCH_LIMIT' or scheduled)
        if result.returncode and not known_checkpoint:
            raise RuntimeError('Generation paused/failed; checkpoint preserved. Restore service before retrying ERROR records.')
        if result.returncode==2 and scheduled:
            print('All attempts saved; ERROR records remain. Judge will score only OK records, not treat errors as PASS.',flush=True)
    finally:export_checkpoint()

def judge_configuration(method):
    if method not in METHODS:raise ValueError('Invalid method')
    if method!=ABLATION_CONFIG or ACTION=='generate_only':
        print('Judge not selected:',method);return
    destination=OUT/'runs'/method
    if not (destination/'run_manifest.json').exists():raise RuntimeError('Generate this configuration first')
    args=[PYTHON,'-u','đo/evaluate_project.py','judge','--run',str(destination),'--allow-llm']
    if MAX_NEW_JUDGMENTS is not None:args+=['--max-new-judgments',str(MAX_NEW_JUDGMENTS)]
    try:
        with model_environment('judge'):result=subprocess.run(args,cwd=ROOT,check=False)
        if result.returncode:raise RuntimeError('Judge failed; answers and completed scores preserved')
        records=[json.loads(line) for line in (destination/'responses.jsonl').read_text(encoding='utf8').splitlines() if line.strip()]
        print(method,'AI_JUDGED',sum(r.get('judge',{}).get('status')=='AI_JUDGED' for r in records),'of',len(records),'saved attempts')
    finally:export_checkpoint()

## 5. Chạy duy nhất cấu hình đã chọn
Nếu ACTION=judge_only, cell này không sinh lại đáp án.

In [ ]:
run_configuration(ABLATION_CONFIG)

## 6. Chấm riêng cấu hình đó bằng Gemini 3.5
Nếu ACTION=generate_only, cell này được bỏ qua. Câu đã AI_JUDGED không được chấm lại.

In [ ]:
judge_configuration(ABLATION_CONFIG)

## 7. Bảng kết quả riêng và ZIP của cấu hình này

In [ ]:
destination=OUT/'runs'/ABLATION_CONFIG
if (destination/'run_manifest.json').exists():
    run([PYTHON,'-u','đo/evaluate_project.py','report','--runs',str(destination),'--out',str(OUT/'comparison')],cwd=ROOT)
    from IPython.display import Markdown
    display(Markdown((OUT/'comparison/comparison.md').read_text(encoding='utf8')))
export_checkpoint()

## 8. Dọn dịch vụ sau cấu hình đã chọn

In [ ]:
if CLEANUP_OLLAMA:
    # Chỉ chạy khi kết thúc toàn bộ phép đo trong phiên này; không chạy giữa các phase.
    if OWN_OLLAMA is not None and OWN_OLLAMA.poll() is None:
        OWN_OLLAMA.terminate()
        try:OWN_OLLAMA.wait(timeout=10)
        except subprocess.TimeoutExpired:
            OWN_OLLAMA.kill(); OWN_OLLAMA.wait()
    if OLLAMA_LOG is not None: OLLAMA_LOG.close()
    print('Đã dọn Ollama do notebook này tạo; không dừng dịch vụ được tái sử dụng.')
else:
    print('Giữ Ollama để chạy cell cấu hình tiếp theo. Đặt CLEANUP_OLLAMA=True rồi chạy cell này khi kết thúc.')
